## Load the Dataset

In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = dataset["train"].to_pandas()

df.head()

README.md: 0.00B [00:00, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…):   0%|          | 0.00/26.0M [00:00<?, ?B/s]

(…)set-tickets-german_normalized_50_5_2.csv: 0.00B [00:00, ?B/s]

dataset-tickets-multi-lang-4-20k.csv:   0%|          | 0.00/18.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


In [3]:
texts = df["body"].fillna("").tolist()

## TF-IDF Search

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(texts)

In [5]:
def search_tfidf(query, top_k=5):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices][["body"]].copy()
    results["score"] = similarities[top_indices]

    return results

In [6]:
search_tfidf("My internet is not working")

,body,score
43328,I am facing difficulties with medical data enc...,0.369616
49090,The internet connection was lost during the me...,0.328430
50914,The internet connection was lost during our me...,0.324739
52574,The encryption process has stopped working,0.319220
13420,Is it possible to get detailed information on ...,0.307803


## Embedding Search

In [9]:
# Create a sample
df_sample = df.sample(n=3000, random_state=42).reset_index(drop=True)

texts = df_sample["body"].fillna("").tolist()

print(f"Using {len(texts)} tickets for embedding search")

Using 3000 tickets for embedding search


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

In [11]:
import numpy as np

# Save embeddings
np.save("ticket_embeddings.npy", embeddings)
df_sample.to_csv("ticket_sample.csv", index=False)

print("Embeddings and sample saved!")

Embeddings and sample saved!


In [12]:
# Load embeddings
import numpy as np
import pandas as pd

embeddings = np.load("ticket_embeddings.npy")
df_sample = pd.read_csv("ticket_sample.csv")

texts = df_sample["body"].fillna("").tolist()

print("Embeddings loaded!")

Embeddings loaded!


In [13]:
# Embedding search function

from sklearn.metrics.pairwise import cosine_similarity

def search_embeddings(query, top_k=5):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(query_embedding, embeddings).flatten()

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = df_sample.iloc[top_indices][["body"]].copy()
    results["score"] = similarities[top_indices]

    return results

In [14]:
#Testing

search_embeddings("My internet is not working")

,body,score
2117,Customers have encountered connectivity proble...,0.417662
164,"Dear Support Team,\n\nOur marketing department...",0.386030
2081,"Dear Customer Support, our agency is encounter...",0.384220
2103,"The digital tools of the marketing agency, inc...",0.345844
1126,Facing occasional service outages,0.345618


## Hybrid Search

In [15]:
# Build TF-IDF on the same sample used for embeddings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sample_texts = df_sample["body"].fillna("").tolist()

sample_vectorizer = TfidfVectorizer(stop_words="english")
sample_tfidf_matrix = sample_vectorizer.fit_transform(sample_texts)

print("Sample TF-IDF matrix created!")

Sample TF-IDF matrix created!


In [16]:
def search_hybrid(query, top_k=5, alpha=0.5):
    # TF-IDF scores on sample
    query_tfidf = sample_vectorizer.transform([query])
    tfidf_scores = cosine_similarity(query_tfidf, sample_tfidf_matrix).flatten()

    # Embedding scores on same sample
    query_embedding = model.encode([query])
    embedding_scores = cosine_similarity(query_embedding, embeddings).flatten()

    # Combine scores
    hybrid_scores = alpha * tfidf_scores + (1 - alpha) * embedding_scores

    top_indices = hybrid_scores.argsort()[-top_k:][::-1]

    results = df_sample.iloc[top_indices][["body"]].copy()
    results["tfidf_score"] = tfidf_scores[top_indices]
    results["embedding_score"] = embedding_scores[top_indices]
    results["hybrid_score"] = hybrid_scores[top_indices]

    return results

In [17]:
# Testing
search_hybrid("My internet is not working")

,body,tfidf_score,embedding_score,hybrid_score
164,"Dear Support Team,\n\nOur marketing department...",0.117612,0.386030,0.251821
2673,The laptop and network connectivity unexpected...,0.159759,0.335663,0.247711
177,"The dashboard is loading slowly, which might b...",0.234998,0.252906,0.243952
45,I hope this message finds you well. I am writi...,0.123287,0.312663,0.217975
2188,Our digital marketing campaign unexpectedly ce...,0.203004,0.222049,0.212527
